# Synthèse du TP2 : stocker avec MongoDB, traiter avec Spark

BigData Commerce Analytics, B2 Sup de Vinci, 26/11/2026.

Cette synthèse résume ce qu'on a fait pendant le TP2, à partir de deux notebooks :
- `notebooks/TP2_Atelier_1_ingestion_api_mongodb.ipynb` : récupérer les données et les **stocker** dans MongoDB ;
- `notebooks/TP2_Atelier_2_Traitement_pyspark.ipynb` : **traiter** ces données avec Spark et calculer les indicateurs.

## 1. L'idée générale du TP

Au TP1, on avait exploré les données avec pandas, sur un seul fichier CSV chargé en mémoire. Ça marche bien pour 800 lignes, mais pas pour des millions. Le TP2 met en place une chaîne de traitement « Big Data », avec un outil pour chaque rôle :

```
API DummyJSON + CSV des commandes
          │  (Python : requests, pandas)
          ▼
      MongoDB  ──────────►  stockage des données brutes (collections raw_*)
          │  (connecteur Mongo-Spark)
          ▼
    Spark (PySpark) ─────►  traitement en parallèle, calcul des indicateurs
          │  (toPandas)
          ▼
   resultats/*.csv  ─────►  graphiques du jour 3
```

- **MongoDB** est une base de données NoSQL : elle range des documents JSON, ce qui tombe bien puisque l'API renvoie du JSON.
- **Spark** est un moteur de calcul distribué : il découpe les données en morceaux et les traite en parallèle sur plusieurs cœurs (ou plusieurs machines sur un vrai cluster).

## 2. Atelier 1 : ingestion dans MongoDB

### 2.1 Connexion à MongoDB

MongoDB tourne dans un conteneur Docker sur le port 27017. On s'y connecte depuis Python avec `pymongo` (`MongoClient`), on teste le serveur avec `ping` (réponse `{'ok': 1.0}`, version 7.0) et on travaille dans la base `commerce`.

### 2.2 Ingestion des 8 ressources de l'API

On télécharge les 8 ressources de DummyJSON (`products`, `carts`, `users`, `posts`, `comments`, `recipes`, `quotes`, `todos`) avec `requests`. Le paramètre `limit=0` est important : sans lui, l'API ne renvoie que 30 éléments.

Chaque ressource est stockée dans une collection `raw_<ressource>`. Le préfixe `raw_` veut dire « brut » : on garde les données telles que l'API les donne, sans les modifier.

Pour pouvoir relancer le notebook sans créer de doublons, on **vide la collection avant d'insérer** (`delete_many({})` puis `insert_many`).

**Contrôle** : pour chaque ressource, on compare le total annoncé par l'API, le nombre de documents reçus et le nombre réellement stocké dans MongoDB. Tout correspond, soit **2 959 documents** au total.

On remarque aussi que MongoDB ajoute un champ `_id` à chaque document : c'est un identifiant technique (ObjectId), différent du champ `id` qui vient de l'API.

### 2.3 Import du fichier des commandes

Le fichier `data/processed/commandes_ecommerce.csv` (800 lignes, 16 colonnes) est importé dans `raw_commandes`. Une ligne correspond à **un produit dans un panier**, donc un panier occupe plusieurs lignes.

Deux précautions avant l'insertion :
- convertir `order_date` en vraie date avec `pd.to_datetime`, sinon MongoDB la stocke comme du texte ;
- remplacer les `NaN` de pandas (surtout dans `brand`) par `None`, qui devient `null` dans MongoDB.

**Contrôle CSV / MongoDB** : on retrouve les mêmes valeurs des deux côtés.

| Contrôle | Valeur |
|---|---:|
| Nombre de documents | 800 |
| Paniers distincts (`cart_id`) | 208 |
| Somme de `discounted_line_total` | 3 456 709,58 |
| Lignes sans marque | 377 |

### 2.4 Simulation des paiements

Les données DummyJSON n'ont pas de paiements, on les simule pour avoir un vrai chiffre d'affaires encaissé.

Les règles choisies :
- **un paiement par panier** (et pas par ligne, sinon on facturerait plusieurs fois le même panier) : 208 paiements ;
- le montant est `cart_discounted_total` (le prix payé après remise), pris **une seule fois** par panier, car il est répété sur chaque ligne ;
- le statut est tiré au hasard : **70 % PAID, 20 % REFUSED, 10 % PENDING**, avec une graine (`random.Random(42)`) pour obtenir le même résultat à chaque exécution ;
- `amount_collected` vaut le montant si le paiement est PAID, 0 sinon ;
- un **index unique sur `cart_id`** empêche MongoDB d'accepter deux paiements pour le même panier.

Avant de créer les paiements, on a vérifié que chaque panier n'a qu'un client, un montant et une commande (0 panier incohérent). Les paniers sont regroupés directement dans MongoDB avec une agrégation (`$group`, `$first`, `$sum`).

**Contrôles après insertion** : 208 paiements, 0 doublon, tous les paniers ont un paiement, seulement les 3 statuts prévus, et surtout **la somme des 208 paiements est égale à la somme des 800 lignes** (3 456 709,58). Aucun montant n'a donc été perdu ni compté deux fois.

Une erreur corrigée en route : au début, on prenait la date la plus récente du panier comme date de paiement. Comme les dates sont tirées au hasard, le maximum tombait presque toujours en fin d'année et créait une fausse hausse des ventes en décembre. On prend maintenant la date de la première ligne.

### 2.5 Résultat de la simulation

| Statut | Paiements | Part | Montant |
|---|---:|---:|---:|
| PAID | 150 | 72,1 % | 2 669 235,61 |
| REFUSED | 37 | 17,8 % | 423 555,75 |
| PENDING | 21 | 10,1 % | 363 918,22 |
| **Total** | **208** | **100 %** | **3 456 709,58** |

- Les proportions sont proches de 70 / 20 / 10 %. L'écart vient du hasard sur seulement 208 tirages.
- **Chiffre d'affaires encaissé : 2 669 235,61**, soit 77,2 % du montant des paniers. 787 473,97 ne sont pas encaissés.
- Taux de refus : 17,79 %.

## 3. Atelier 2 : traitement avec PySpark

### 3.1 Préparer Spark

On crée une **SparkSession** en mode `local[*]` : Spark tourne sur notre PC mais utilise tous ses cœurs (16 ici), comme un mini-cluster. On ajoute le **connecteur MongoDB** (`mongo-spark-connector_2.13:11.1.0`) pour que Spark puisse lire les collections. Versions : PySpark 4.0.1, Java 17.

Spark propose aussi une interface web, **Spark UI** (http://localhost:4040), qui montre ce qu'il exécute.

### 3.2 Lire MongoDB avec Spark

`spark.read.format("mongodb")` lit `raw_commandes` (800 lignes) et `raw_payments` (208 lignes) sous forme de **DataFrames Spark**. On retrouve les mêmes volumes que dans MongoDB.

Le schéma (noms et types des colonnes) n'est écrit nulle part : le connecteur le **devine** en lisant les documents. On supprime la colonne `_id` avec `.drop("_id")` car elle n'a pas de sens métier.

### 3.3 Transformations et actions

C'est la notion la plus importante de Spark. Il y a deux types d'opérations :

| Type | Exemples utilisés | Ce que ça fait |
|---|---|---|
| **Transformation** | `select`, `filter`, `withColumn`, `drop`, `groupBy` + `agg`, `orderBy`, `join` | Décrit un nouveau DataFrame. **Rien n'est calculé**, Spark note juste l'étape dans son plan. |
| **Action** | `show`, `count`, `take`, `first`, `toPandas` | Demande un résultat. **C'est là que Spark lit les données** et exécute tout le plan. |

Exemple du TP : on garde les lignes avec au moins 2 articles (`filter`), on calcule la remise de chaque ligne (`withColumn`), on regroupe par catégorie (`groupBy`), puis on affiche (`show`).

### 3.4 La lazy evaluation (évaluation paresseuse)

On l'a vérifié avec Spark UI ouvert à côté du notebook :

- **Cellule A** : 5 transformations sur les paiements (garder les PAID, créer une colonne mois, regrouper, sommer, trier) + `explain()` pour afficher le plan. Résultat : **0 job** dans Spark UI.
- **Cellule B** : un simple `show()`. Résultat : **2 jobs** apparaissent.

Conclusion : Spark attend qu'on lui demande un résultat pour travailler. L'avantage, c'est qu'il connaît tout le traitement à l'avance et peut l'**optimiser**. Dans le plan d'`explain()`, on voit par exemple qu'il ne lit que les 3 colonnes utiles (`status`, `payment_date`, `amount`) au lieu de toute la collection.

### 3.5 Partitions, jobs, stages et tasks

Un DataFrame Spark est découpé en **partitions** (des morceaux de données). Chaque partition peut être traitée par un cœur différent : c'est ce qui permet le parallélisme.

Ce qu'on a observé :
- les collections lues depuis MongoDB n'ont qu'**1 partition** (800 lignes, c'est trop peu pour que Spark découpe) ;
- après `repartition(4)`, on a **4 partitions de 200 lignes**, et Spark UI montre 4 tasks qui tournent en parallèle.

Le vocabulaire :
- **Task** : le traitement d'une partition par un cœur (1 partition = 1 task).
- **Stage** : un groupe de tasks qui n'ont pas besoin d'échanger de données. Un `groupBy`, un `orderBy` ou un `repartition` oblige Spark à redistribuer les données entre partitions (on appelle ça un **shuffle**), ce qui crée un nouveau stage.
- **Job** : l'ensemble des stages lancés par une action.

En résumé : **une action → un ou plusieurs jobs → des stages (coupés à chaque shuffle) → une task par partition.**

Les temps mesurés (quelques millisecondes) ne veulent pas dire grand-chose ici : avec si peu de données, organiser le travail prend plus de temps que le calcul lui-même. Spark devient intéressant avec de gros volumes.

## 4. Atelier 3 : les 4 indicateurs

Deux règles à respecter :
- ne **jamais additionner `cart_total` ou `cart_discounted_total`** sur les lignes, car ils sont répétés sur chaque ligne du panier ;
- le **chiffre d'affaires encaissé** se calcule uniquement avec les paiements **PAID**.

Les deux DataFrames de base sont gardés en mémoire avec `cache()` car ils servent à tous les indicateurs : Spark n'a pas à relire MongoDB à chaque fois.

| Indicateur | Calcul Spark | Résultat principal |
|---|---|---|
| 1. CA par catégorie | `groupBy("category")` + somme de `discounted_line_total`, et `join` avec les paiements PAID pour le CA encaissé | `vehicle` = 48,7 % du montant des paniers. Les 4 premières catégories font environ 91 %. |
| 2. Top 10 clients | paiements PAID, `groupBy("user_id")`, tri décroissant, `limit(10)` | Le n°1 (`user_id` 30) a payé 155 507,16. Le top 10 pèse 46,2 % du CA encaissé. |
| 3. Taux de refus | nombre de REFUSED / nombre total de paiements | 37 / 208 = **17,79 %** |
| 4. Évolution mensuelle | paiements PAID, colonne `mois` avec `date_format`, `groupBy("mois")` | De 54 081,45 (juillet) à 432 281,84 (février). |

Les 4 résultats sont exportés en CSV dans `resultats/` avec `toPandas()` pour les graphiques du jour 3.

### Ce qu'il faut retenir des résultats

- **Le CA dépend surtout du prix des produits** : une voiture vaut des dizaines de milliers, un mascara quelques euros. Peu d'articles chers suffisent à faire la moitié du chiffre.
- **Le top 10 clients est en fait un top 10 des paniers** : dans DummyJSON, chaque client n'a qu'un seul panier. On ne peut donc rien dire sur la fidélité.
- **L'évolution mensuelle n'est pas une vraie saisonnalité** : les dates sont générées pour l'exercice, et les grosses différences entre les mois viennent de quelques paniers très chers.

## 5. Difficultés rencontrées (Windows)

Le TP est prévu sous Ubuntu. Sous Windows, il a fallu :
- installer **Java 17**, car Java 25 n'est pas pris en charge par Spark 4.0 ;
- créer l'environnement Python dans `C:\venvs\bigdata`, car l'accent dans le nom d'utilisateur empêchait Spark de trouver ses fichiers ;
- définir `SPARK_LOCAL_HOSTNAME=localhost`, car le nom du PC contient aussi un accent ;
- installer `winutils.exe` et `hadoop.dll` (`HADOOP_HOME`), dont Hadoop a besoin sous Windows ;
- exporter les CSV avec `toPandas()` plutôt qu'avec `df.write` de Spark, qui demande encore d'autres outils Hadoop sous Windows.

## 6. Bilan

| Notion | Ce qu'on a appris |
|---|---|
| Collection MongoDB | Sert à **stocker** des documents JSON, chacun peut avoir sa propre structure. Les données restent sur disque. |
| DataFrame Spark | Sert à **traiter** : un tableau avec des colonnes typées, découpé en partitions, qui n'existe que pendant la session Spark. |
| Transformation / action | La transformation prépare, l'action exécute. |
| Lazy evaluation | Spark ne calcule rien avant une action, ce qui lui permet d'optimiser tout le plan. |
| Partition / task / stage / job | Les données sont découpées en partitions, chaque partition donne une task, les shuffles séparent les stages, et une action lance un job. |
| Contrôles de cohérence | À chaque étape, on compare les chiffres (API, CSV, MongoDB, Spark) pour être sûr de ne rien perdre ni compter deux fois. |
| CA encaissé | Un panier n'est pas une vente : seuls les paiements PAID comptent (2 669 235,61 et pas 3 456 709,58). |

**Pour le jour 3** : utiliser les CSV de `resultats/` pour les graphiques, et s'occuper des données personnelles. `raw_users` contient en clair des données très sensibles (mot de passe, numéro de sécurité sociale, IBAN, adresse...). Il faudra ne garder que les champs utiles, pseudonymiser `user_id` et `username`, et protéger MongoDB par un mot de passe.